this notebook contains the cleaning process for the crime data and the preliminary analysis.

# 1.0 Load data and packages

In [ ]:
# load the packages
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import NearestNeighbors
from scipy.stats import chi2_contingency

In [ ]:
# load the crime data
crime_df = pd.read_csv('crime_data.csv')

In [ ]:
# Load schools
school_df = pd.read_csv('results.csv')

In [ ]:
# Load UK boundary
uk_boundary = gpd.read_file('boundary.geojson')

# 2.0 Data Preview

## 2.1 Cime Data

In [ ]:
# view crime data
crime_df.head(3)

In [ ]:
# check the shape of the data
crime_df.shape

In [ ]:
# check data 
crime_df.info()

In [ ]:
# confirm null values
crime_df.isnull().sum()

geometries for over 4500 crimes are missing.

In [ ]:
# check the columns
crime_df.columns

In [ ]:
# check the columns
crime_df.lsoa_name.nunique()

In [ ]:
crime_df.lsoa.unique()

In [ ]:
crime_df.lsoa_name.unique()

In [ ]:
crime_df.lsoa_name1.unique()

In [ ]:
crime_df.lsoa1.unique()

In [ ]:
# confirm unique date time
crime_df.cb_date_added.unique()

In [ ]:
# confirm unique date
crime_df.date_recorded.unique()

In [ ]:
# confirm the maximum year
crime_df.date_recorded.max()

In [ ]:
# check the crime types
crime_df.offence.unique()

## 2.2 School Data

In [ ]:
# check data structure
school_df.shape

In [ ]:
# check teh datatypes
school_df.info()

In [ ]:
# confirm column names
school_df.columns

In [ ]:
# change name
school_df = school_df.rename(columns={'EstablishmentName': 'school_name'})

## 2.3 Boundary

In [ ]:
# confirm structure
uk_boundary.shape

In [ ]:
# check the datatypes
uk_boundary.info()

In [ ]:
# Filter for bradford
bradford_boundary = uk_boundary[
uk_boundary['LSOA21NM'].str.contains('Bradford', case=False, na=False)
]

# 3.0 Data Cleaning

## 3.1 Boundary data

In [ ]:
# Check remaining columns
bradford_boundary.columns

In [ ]:
bradford_boundary['LSOA21NM'].unique()

In [ ]:
# Drop some other columns
bradford_boundary = bradford_boundary.drop(
    ['FID', 'BNG_E', 'LSOA21NMW', 'BNG_N', 'GlobalID'], axis=1)

In [ ]:
# Rename the columns
bradford_boundary = bradford_boundary.rename(
    columns={
        'LSOA21CD': 'lsoa',
        'LSOA21NMW': 'lsoa_name',
        'LONG': 'bound_long',
        'LAT': 'bound_lat'}
)

In [ ]:
# Check coordinates
print(bradford_boundary.crs)

In [ ]:
# Transform to British National Grid (Meters)
bradford_boundary = bradford_boundary.to_crs(epsg=27700)

# confoirm change
print(bradford_boundary.crs)

In [ ]:
# save the cleaned coundary data
bradford_boundary.to_file('bradford_boundary.geojson', driver='GeoJSON')

## 3.2 School data

Crime data is from 2019 to 2020; I will remove the schools that closed before this time. I am equally aware that some of these schools were repurposed, I will keep open date as-is.

In [ ]:
# convert to time format
school_df['CloseDate'] = pd.to_datetime(school_df['CloseDate'], format='%d-%m-%Y', errors='coerce')

In [ ]:
# define cutoff 
cutoff_close = pd.to_datetime('2018-12-31', format='%Y-%m-%d')

# filter the cutoff
school_df = school_df[(school_df['CloseDate'] > cutoff_close) | (school_df['CloseDate'].isna())].copy()

In [ ]:
# remove duplicate schools using locations
school_df = school_df.drop_duplicates(subset=['Easting', 'Northing'], keep='last').copy()

In [ ]:
# Filter the cutoff: Keep schools closing after the cutoff or schools that are still open
school_df = school_df[(school_df['CloseDate'] > cutoff_close) | (school_df['CloseDate'].isna())].copy()

In [ ]:
school_df.school_name.nunique()

I will use schoolcapacity as a proxy for the population of the schools. This is because GIAS does not have complete number of pupils for all the schools, as what is obtaianable is current census.

In [ ]:
school_df['SchoolCapacity'] = pd.to_numeric(school_df['SchoolCapacity'], errors='coerce')

In [ ]:
# replace nan on the schoolcapacity
cap_med = school_df['SchoolCapacity'].median()
school_df['SchoolCapacity'] = school_df['SchoolCapacity'].fillna(cap_med)

In [ ]:
# Drop the unnecessary columns
school_df = school_df.drop(
    ['UKPRN', 'Trusts (name)', 'SchoolSponsors (name)', 'CensusDate', 'FEHEIdentifier','SchoolWebsite',
     'InspectorateReport', 'DateOfLastInspectionVisit', 'LSOA (name)', 'PropsName', 'UPRN',
     'PropsType (name)', 'LSOA (code)', 'FurtherEducationType (name)', 'EstablishmentStatus (name)', 'OpenDate',
    'ReasonEstablishmentClosed (name)', 'CloseDate', 'PhaseOfEducation (name)', 'NumberOfPupils'],
    axis=1)

In [ ]:
# convert easting and northing to meter
geometry=gpd.points_from_xy(
        school_df['Easting'], school_df['Northing'])

# transform to british grid
sec_gdf = gpd.GeoDataFrame(school_df, geometry=geometry, crs="EPSG: 27700")

In [ ]:
print(sec_gdf.crs)

Pseudonymise the school names

In [ ]:
# get the unique names
unique_names = sec_gdf['school_name'].unique()

# initialise empty dictionary
anon = {}

# loop for each school to create a schxx name
for idx, name in enumerate(unique_names):
    anon[name] = f"school{idx+1:02d}"

In [ ]:
# map it to your dataframe column
sec_gdf['sch_id'] = sec_gdf['school_name'].map(anon)

In [ ]:
# save the clean school data
sec_gdf.to_csv('sec_gdf.csv', index=False)
sec_gdf.to_file('sec_gdf.geojson', driver='GeoJSON')

## 3.3 Crime data

### 3.3.1 standardise columns

In [ ]:
# remove non bradford crimes
crime_df = crime_df[crime_df['lsoa_name'].str.contains('Bradford', case=False, na=False)]

In [ ]:
# check null data
crime_df.info()

post code has 1734 missing values. The column is not needed too

In [ ]:
# drop unnecessary columns
crime_df = crime_df.drop(
    ['Unnamed: 0', 'cb_date_added', 'Index', 'postcode_district',
     'postcode', 'ho_class', 'lsoa1', 'lsoa_name1'], axis=1)

In [ ]:
# convert easting and northing to meter
# creat geomerty point
geometry=gpd.points_from_xy(
        crime_df['easting'], crime_df['northing'])

# transform to british grid
crime_gdf = gpd.GeoDataFrame(crime_df, geometry=geometry, crs="EPSG: 27700")

In [ ]:
# check the grid
print(crime_gdf.crs)

### 3.3.2 Handle time

Date Earliest: the earliest possible date and time that the crime could have started.\
Date Latest: the latest possible date and time that the crime could have been committed or that victim found out. Mostly relevant if a crime had happened but the household had not reported it on time.\
Recorded Date: the official administrative timestamp when the reporting police officer decided a crime occurred and permanently registered it into the system database.

Of the three, i think the data earliest identifies the time the crime is committed, so it helps with the spcific time a crime occurs for our analysis

In [ ]:
# inspect the date earliest and latest
date_latest = crime_gdf[crime_gdf['date_latest'] >= '2020-12-31 00:00:00']
date_earliest = crime_gdf[crime_gdf['date_earliest'] >= '2020-12-31 00:00:00']

In [ ]:
date_latest

On date earliest, where the date is mislabelled 2029, 2109, their corresponding date recorded is 2019; where the date earlist is 2030, the corresponding date recorded is 2020.

For date latest, inaddition to have similar mislabelling, there is 2023 recorded in 2019, 2300 recorded in 2019, 5019 recorded in 2019, 2359 recorded in 2019, 2049 recorded in 2019.

In [ ]:
print('date latest:', date_latest['date_latest'].unique())
print('date earliest:', date_earliest['date_earliest'].unique())

There are mislabelled dates on the data earliest, and the date latest is much different. Fix the dates.

In [ ]:
# Create a replacement dictionary for the messy year patterns
wrong_year = {
    '2109-': '2019-',
    '2039-': '2019-',
    '2029-': '2019-',
    '2030-': '2020-',
    '2023-': '2019-',
    '2300-': '2019-',
    '5019-': '2019-',
    '2049-': '2019-',
    '2359-': '2019-'
}

# replace in the data
crime_gdf['date_earliest'] = crime_gdf['date_earliest'].replace(wrong_year, regex=False)
crime_gdf['date_latest'] = crime_gdf['date_latest'].replace(wrong_year, regex=False)

In [ ]:
# fix the datetime formats
time_cols = ['date_recorded', 'date_earliest', 'date_latest']

for col in time_cols:
    crime_gdf[col] = pd.to_datetime(
        crime_gdf[col], errors='coerce')

I am prioritising date earlist as my date when crime happened/reported.

In [ ]:
#check missing time
missing_earliest = crime_gdf[crime_gdf['date_earliest'].isna()]

In [ ]:
# drop na on the date column
crime_gdf = crime_gdf.drop(missing_earliest.index, axis=0)

### 3.3.3 Derive time columns and seasons

In [ ]:
# extract the year, month and day
crime_gdf['year'] = crime_gdf['date_earliest'].dt.year
crime_gdf['month'] = crime_gdf['date_earliest'].dt.month
crime_gdf['day'] = crime_gdf['date_earliest'].dt.day
crime_gdf['date_only'] = crime_gdf['date_earliest'].dt.date

In [ ]:
# add name for month and day
crime_gdf['month_name'] = crime_gdf['date_earliest'].dt.month_name()
crime_gdf['day_name'] = crime_gdf['date_earliest'].dt.day_name()

In [ ]:
# add month label
crime_gdf["month_label"] = (
    crime_gdf["year"].astype(str)
    + "-"
    + crime_gdf["month"].astype(str).str.zfill(2)
)

In [ ]:
# extract time
crime_gdf['time'] = crime_gdf['date_earliest'].dt.strftime('%H:%M')

In [ ]:
# convert the hhmm time to minutes
def hhmm_to_minutes(value):
    if pd.isna(value):
        return None

    value = str(value)

    hours, minutes = value.split(":")
    return int(hours) * 60 + int(minutes)

In [ ]:
# map with times
crime_gdf['time_min'] = crime_gdf['time'].apply(hhmm_to_minutes)

In [ ]:
# create the day of week type
def classify_day_type(day):
    if day in ['Sunday', 'Saturday']:
        return 'Weekend'
    else:
        return 'Weekday'

In [ ]:
# map the data
crime_gdf['day_type'] = crime_gdf['day_name'].apply(classify_day_type)

In [ ]:
# keep only weekday crimes
crime_gdf = crime_gdf[~crime_gdf['day_name'].isin(['Saturday', 'Sunday'])].copy()

In [ ]:
# map the order of the days
day_map = {
    'Monday': 0,
    'Tuesday': 1,
    'Wednesday': 2,
    'Thursday': 3,
    'Friday': 4,
    'Saturday': 5,
    'Sunday': 6
}

crime_gdf['day_num'] = crime_gdf['day_name'].str.strip().map(day_map)

In [ ]:
# calculate the nearest hour withi which crime happened
crime_gdf['hour'] = crime_gdf['time_min'] // 60

# add 24 hours to the hour after each day
crime_gdf['weekly_hour'] = (crime_gdf['day_num'] * 24) + crime_gdf['hour']

In [ ]:
# define transition timing
def assign_commute(time_min):
    if 420 <= time_min < 540:
        return 'morning commute'
    elif 900 < time_min <= 1020:
        return 'afternoon commute'
    else:
        return 'non commute hours'

# map the column to time_min
crime_gdf['transition'] = crime_gdf['time_min'].apply(assign_commute)

In [ ]:
# Classify each date into the broad academic term
def classify_season_period(date):

    date_str = date.strftime('%Y-%m-%d')

    # 2017
    if '2017-01-01' <= date_str <= '2017-04-23':
        return 'spring'

    elif '2017-04-24' <= date_str <= '2017-08-31':
        return 'summer'

    elif '2017-09-01' <= date_str <= '2017-12-31':
        return 'autumn'

    # 2018
    elif '2018-01-01' <= date_str <= '2018-04-15':
        return 'spring'

    elif '2018-04-16' <= date_str <= '2018-08-31':
        return 'summer'

    elif '2018-09-01' <= date_str <= '2018-12-31':
        return 'autumn'

    # 2019
    elif '2019-01-01' <= date_str <= '2019-04-28':
        return 'spring'

    elif '2019-04-29' <= date_str <= '2019-08-31':
        return 'summer'

    elif '2019-09-01' <= date_str <= '2019-12-31':
        return 'autumn'

    # 2020
    elif '2020-01-01' <= date_str <= '2020-04-19':
        return 'spring'

    elif '2020-04-20' <= date_str <= '2020-08-31':
        return 'summer'

In [ ]:
# Create broad academic term column
crime_gdf['season'] = (
    crime_gdf['date_only']
    .apply(classify_season_period)
)

#### 3.3.3.1 Add the term time and holidays

I will use Bingley Grammar School calendar for the holidays. See the reference:

2017/2018: https://bso.bradford.gov.uk/council/schools/holidaycalendarview.aspx?dfes=3805400&accyear=2017 \
2018/2019: https://bso.bradford.gov.uk/council/schools/holidaycalendarview.aspx?dfes=3805400&accyear=2018 \
2019/2020: https://bso.bradford.gov.uk/Schools/holidayCalendarView.aspx?accyear=2019&d=3805400

In [ ]:
# define school holiday periods
# Spring half-term - statutory days only
spring_half_term = [
    # 2017/18
    '2018-02-12', '2018-02-13', '2018-02-14',
    # 2018/19
    '2019-02-18', '2019-02-19', '2019-02-20',
    # 2019/20
    '2020-02-17', '2020-02-18', '2020-02-19'
]


# Easter
easter = [
    # 2017/18
    '2018-04-03', '2018-04-04', '2018-04-05', '2018-04-06', '2018-04-09', '2018-04-10', '2018-04-11', '2018-04-12', '2018-04-13',
    # 2018/19
    '2019-04-15', '2019-04-16', '2019-04-17', '2019-04-18', '2019-04-23', '2019-04-24', '2019-04-25', '2019-04-26',
    # 2019/20
    '2020-04-06', '2020-04-07', '2020-04-08', '2020-04-09', '2020-04-14', '2020-04-15', '2020-04-16', '2020-04-17'
]

# Summer half-term - statutory days only
summer_half_term = [
    # 2017/18
    '2018-05-28', '2018-05-29', '2018-05-30', '2018-05-31', '2018-06-01',
    # 2018/19
    '2019-05-28', '2019-05-29', '2019-05-30', '2019-05-31',
    # 2019/20
    '2020-05-26', '2020-05-27', '2020-05-28', '2020-05-29'
]

# Summer holiday
summer = [
    # Summer 2017
    '2017-08-01', '2017-08-02', '2017-08-03', '2017-08-04', '2017-08-07', '2017-08-08', '2017-08-09', '2017-08-10', '2017-08-11',
    '2017-08-14', '2017-08-15', '2017-08-16', '2017-08-17', '2017-08-18', '2017-08-21', '2017-08-22', '2017-08-23', '2017-08-24',
    '2017-08-25', '2017-08-29', '2017-08-30', '2017-08-31', '2017-09-01',
    # Summer 2018
    '2018-07-26', '2018-07-27', '2018-07-30', '2018-07-31', '2018-08-01', '2018-08-02', '2018-08-03', '2018-08-06', '2018-08-07',
    '2018-08-08', '2018-08-09', '2018-08-10', '2018-08-13', '2018-08-14', '2018-08-15', '2018-08-16', '2018-08-17', '2018-08-20',
    '2018-08-21', '2018-08-22', '2018-08-23', '2018-08-24', '2018-08-28', '2018-08-29', '2018-08-30', '2018-08-31',
    # Summer 2019
    '2019-07-22', '2019-07-23', '2019-07-24', '2019-07-25', '2019-07-26', '2019-07-29', '2019-07-30', '2019-07-31', '2019-08-01',
    '2019-08-02', '2019-08-05', '2019-08-06', '2019-08-07', '2019-08-08', '2019-08-09', '2019-08-12', '2019-08-13', '2019-08-14',
    '2019-08-15', '2019-08-16', '2019-08-19', '2019-08-20', '2019-08-21', '2019-08-22', '2019-08-23', '2019-08-27', '2019-08-28',
    '2019-08-29', '2019-08-30',

    # Summer 2020
    '2020-07-21', '2020-07-22', '2020-07-23', '2020-07-24', '2020-07-27', '2020-07-28', '2020-07-29', '2020-07-30', '2020-07-31'
]

# Autumn half-term - statutory days only
autumn_half_term = [
    # 2017/18
    '2017-10-20', '2017-10-23', '2017-10-24', '2017-10-25',
    # 2018/19
    '2018-10-22', '2018-10-23', '2018-10-24',
    # 2019/20
    '2019-10-28', '2019-10-29', '2019-10-30'
]

# Christmas - statutory days only
christmas = [
    # 2017/18
    '2017-12-21', '2017-12-22', '2017-12-27', '2017-12-28', '2017-12-29', '2018-01-02', '2018-01-03',
    # 2018/19
    '2018-12-24', '2018-12-27', '2018-12-28', '2018-12-31', '2019-01-02', '2019-01-03',
    # 2019/20
    '2019-12-23', '2019-12-24', '2019-12-27', '2019-12-30', '2019-12-31', '2020-01-02', '2020-01-03'
]

# Training / INSET days
inset_days = [
    # 2017/18
    '2017-09-04', '2017-09-05', '2017-09-27', '2018-02-09', '2018-03-29',
    # 2018/19
    '2018-09-03', '2018-09-04', '2018-09-20', '2019-01-04', '2019-04-12',
    # 2019/20
    '2019-09-02', '2019-09-03', '2019-09-25', '2020-02-14', '2020-07-20'
]

# Bank holidays shown on the Bingley calendars
bank_holidays = [
    # 2017/18
    '2017-08-28', '2017-12-25', '2017-12-26', '2018-01-01', '2018-03-30', '2018-04-02', '2018-05-07', '2018-08-27',
    # 2018/19
    '2018-12-25', '2018-12-26', '2019-01-01', '2019-04-19', '2019-04-22', '2019-05-06', '2019-05-27', '2019-08-26',
    # 2019/20
    '2019-12-25', '2019-12-26', '2020-01-01', '2020-04-10', '2020-04-13', '2020-05-08', '2020-05-25'
]

# Occasional days
occasional_days = [
    # 2017/18
    '2017-10-26', '2017-10-27', '2018-02-15', '2018-02-16',
    # 2018/19
    '2018-10-25', '2018-10-26', '2019-02-21', '2019-02-22',
    # 2019/20
    '2019-10-31', '2019-11-01', '2020-02-20', '2020-02-21'
]

In [ ]:
# Classify weekdays by school-calendar period
def classify_school_period(date_val):

    # Exclude weekends completely
    if date_val.weekday() >= 5:
        return pd.NA

    date_str = date_val.strftime('%Y-%m-%d')

    if date_str in occasional_days:
        return 'occasional days'

    if date_str in autumn_half_term:
        return 'autumn half term'

    elif date_str in christmas:
        return 'christmas'

    elif date_str in spring_half_term:
        return 'Spring half term'

    elif date_str in easter:
        return 'easter'

    elif date_str in summer_half_term:
        return 'summer half term'

    elif date_str in summer:
        return 'summer'

    elif date_str in inset_days:
        return 'inset days'

    elif date_str in bank_holidays:
        return 'bank holiday'

    else:
        return 'Term time'

In [ ]:
# Map the school-period classification across all dates
crime_gdf['term_time_subperiod'] = (
    crime_gdf['date_only']
    .apply(classify_school_period)
)

Calssify/categorise the holiday groups

In [ ]:
# default everything to holiday first
crime_gdf['term_time_type'] = 'holiday'

# overwrite term time
crime_gdf.loc[crime_gdf['term_time_subperiod'] == 'Term time', 'term_time_type'] = 'Term time'

Create iso calendar weeks

In [ ]:
# create and get the week number
crime_gdf['week_number'] = pd.to_datetime(crime_gdf['date_only']).dt.isocalendar().week
crime_gdf['week_padded'] = crime_gdf['week_number'].astype(str).str.zfill(2)

In [ ]:
# pad the year with the week
crime_gdf['year_label'] = pd.to_datetime(crime_gdf['date_only']).dt.isocalendar().year
crime_gdf['year&week'] = crime_gdf['year_label'].astype(str) + "-W" + crime_gdf['week_padded']

In [ ]:
# add school and holiday suffixes to specify days within school week and holiday weeks
# copy the year&week
crime_gdf['year_week'] = crime_gdf['year&week']

# add 's' to school days and 'h' to holidays
crime_gdf.loc[crime_gdf['term_time_type'] == 'school day', 'year_week'] += 's'
crime_gdf.loc[crime_gdf['term_time_type'] != 'school day', 'year_week'] += 'h'

# 4.0 Sample selection

In [ ]:
# view the crimes with years
crime_gdf['year'].value_counts().to_frame()

I will focus on years with sufficient data points, over 2000 records per year: 2019 and 2020

In [ ]:
# save the cleaned crime data including outside commute window
crime_gdf.to_file('crime_gdf.geojson', driver='GeoJSON', index=False)

In [ ]:
# filter data
crime_gdf = (
    crime_gdf[
    (crime_gdf['year'].isin([2019, 2020])) & 
    (crime_gdf['transition'].isin(['morning commute', 'afternoon commute']))
     ]).copy()

In [ ]:
crime_gdf.crime_ref.nunique()

## 4.1 Merge Datasets

In [ ]:
# join the school with the distance
crime_nearest_school = gpd.sjoin_nearest(
    crime_gdf, sec_gdf, how="left",
    exclusive=True,
    distance_col="distance_to_school_m"
)

crime_nearest_school.columns

# 5.0 Exploratory Data Analysis

## 5.1 Overview of crimes

In [ ]:
# set plot layout
fig, ax = plt.subplots(figsize=(12, 10))

# lay the boundaries
bradford_boundary.plot(ax=ax, color='#f2f2f2', edgecolor='#b2b2b2', linewidth=1)

# plot crime points
crime_nearest_school.plot(ax=ax, color='#2C3E50', markersize=1,
                 alpha=0.6, zorder=2, label='Recorded Incident')

# lay the school points
sec_gdf.plot(ax=ax, color='#DC143C', markersize=12,
             marker='^', zorder=3, label='Secondary School')

# labels and titles
ax.set_axis_off()
plt.title("Crime Incidents", fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 5.2 Univariate analysis

Distance to the nearest school and the time of occurence

In [ ]:
crime_nearest_school["distance_to_school_m"].mean()

In [ ]:
# set up side-by-side plotting area
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Plot Spatial Proximity distribution
sns.histplot(crime_nearest_school["distance_to_school_m"], kde=True, ax=axes[0], color="#D62728")
axes[0].set_title("Distance to Nearest School", fontsize=16, fontweight='bold')
axes[0].set_xlabel("Distance (meters)", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Crime count", fontsize=12, fontweight='bold')

# Plot Commute Minutes
sns.histplot(crime_nearest_school["time_min"], bins=60, kde=True, ax=axes[1], color="#D62728")
axes[1].set_xticks([0, 360, 720, 1080, 1440])
axes[1].set_xticklabels(['0:00', '6:00', 'Noon', '18:00', '23:59'], fontsize=10)
axes[1].set_title("Time of Event within School Commute Period", fontsize=16, fontweight='bold')
axes[1].set_xlabel("Time of Event", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Crime count", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# get count by transition
count_df = crime_nearest_school.groupby(
    ['date_only', 'transition']).size().reset_index(name='crime_count')

In [ ]:
# plot the count of crimes
plt.figure(figsize=(10, 5))
sns.histplot(data=count_df, x='crime_count', kde=True, color='#D62728', discrete=True)

plt.title('Crime Distribution within the Commute Period', fontweight='bold', pad=15)
plt.xlabel('')
plt.ylabel('Counts')
plt.tight_layout()
plt.show()

visualise crime type

In [ ]:
# Count records by offence
crime_typelist = (
    crime_nearest_school
    .groupby("offence")
    .size()
    .rename("crime_count")
    .reset_index()
)

# Calculate each offence's share of all counted records
crime_typelist["percentage"] = (
    crime_typelist["crime_count"] / crime_typelist["crime_count"].sum() * 100)

In [ ]:
# Sort so the largest bar appears at the top
crime_typelist = crime_typelist.sort_values("percentage")

# Plot the percentages
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(crime_typelist["offence"], crime_typelist["percentage"])

# Show the percentage at the end of each bar
ax.bar_label(bars, fmt="%.1f%%", padding=3)

ax.set_xlabel("Percentage of recorded crimes")
ax.set_ylabel("Crime type")
ax.set_title("Recorded Crimes by Type")
ax.set_xlim(0, crime_typelist["percentage"].max() * 1.15)

plt.tight_layout()
plt.show()

Violence without injury: 26.1% \
Public order offence 14.8% \
Crimina damage 9.1% \
Violence with injury 8.6% \
Shoplifting 8.4%

Commute window characteristics

In [ ]:
# Variables relevant to your filtered commute dataset
commute_variables = ["term_time_type", "day_name", "season", "month_name", "year_label"]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, var in enumerate(commute_variables):
    cat_order = crime_nearest_school[var].value_counts().index
    
    sns.countplot(x=var, 
        data=crime_nearest_school, ax=axes[i], order=cat_order, 
        hue=var, palette="Blues_r", legend=False)
    
    axes[i].set_title(f"Distribution of {var.replace('_', ' ').title()}")
    axes[i].tick_params(axis="x", rotation=45)

# Remove the unused space
fig.delaxes(axes[-1])
fig.suptitle("Commute Time Window Characteristics", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


top transition features

In [ ]:
# select engineered time suffixes and transitions
indicator_cols = ["transition", "year_week", "week_padded"]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes = axes.flatten()

for i, var in enumerate(indicator_cols):
    # Keep only the top 10 for year_week or week_padded so the labels remain legible
    top_categories = crime_nearest_school[var].value_counts().head(10).index
    
    sns.countplot(x=var, data=crime_nearest_school, 
                  ax=axes[i], order=top_categories, hue=var, 
                  palette="viridis", legend=False)
    
    axes[i].set_title(f"Top Categories: {var.replace('_', ' ').title()}")
    axes[i].set_xlabel(var)
    axes[i].set_ylabel("Incident Count")
    axes[i].tick_params(axis="x", rotation=45)

fig.suptitle("Analysis of Engineered Time Windows and Suffix Profiles", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

Hourly commute

In [ ]:
# select spatial and hourly markers inside the commute periods
numerical_metrics = ["hour", "weekly_hour"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
axes = axes.flatten()

for i, var in enumerate(numerical_metrics):
    sns.histplot(data=crime_nearest_school, x=var, kde=True, ax=axes[i], color="#1F77B4")
    axes[i].set_title(var.replace("_", " ").title())
    axes[i].set_ylabel("Incident Frequency")

fig.suptitle("Spatial and Discrete Hour Commute", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 5.3 Bivariate analysis

Commute windows and distance

In [ ]:
# initialise
plt.figure(figsize=(12, 4))

# Swapped x and y variables to make the boxplot horizontal
sns.boxplot(data=crime_nearest_school, x='distance_to_school_m', y='transition',
            hue='transition', palette=['#D62728', '#1F77B4'], legend=False)

plt.title('Proximity at Commute Windows', fontweight='bold')
plt.xlabel('Distance to School (Meters)', fontweight='bold')
plt.ylabel('Commute Window', fontweight='bold')

# cap distance
# plt.xlim(0, 1500)

plt.tight_layout()
plt.show()


Crimes per day window

In [ ]:
# group by date and commute window to get daily counts
daily_counts = crime_nearest_school.groupby(['date_only', 'transition']).size().reset_index(name='crime_count')

In [ ]:
# plot the boxplot
plt.figure(figsize=(10, 5))
sns.boxplot(data=daily_counts, x='transition', y='crime_count', 
            palette=['#D62728', '#1F77B4'], hue='transition', legend=False)

plt.title('Distribution of Daily Crime Counts Across Commute Windows', fontweight='bold')
plt.xlabel('Commute Window')
plt.ylabel('Number of Crimes per Day')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))

# Added stat="percent" to automatically switch the calculation from count to percentage
sns.countplot(data=crime_nearest_school, x='transition', stat="percent",
              palette=['#D62728', '#1F77B4'], hue='transition', legend=False)

plt.title('Percentage of Total Crimes Across Commute Windows', fontweight='bold')
plt.xlabel('Commute Window', fontweight='bold')
plt.ylabel('Percentage of Crime (%)', fontweight='bold')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

Day composition of the crimes

In [ ]:
# Matrix of day names versus school operation status
day_term_matrix = pd.crosstab(
    index=crime_nearest_school['day_name'], 
    columns=crime_nearest_school['term_time_type'],
    normalize='columns'
) * 100

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(day_term_matrix, annot=True,
            fmt=".1f", cmap="YlGnBu",
            cbar_kws={'label': '% Share'})
plt.title('Weekday Crime Distribution: School Term vs Holiday (%)', fontweight='bold')
plt.ylabel('Day of the Week')
plt.xlabel('Academic Status')
plt.tight_layout()
plt.show()

### Save data

In [ ]:
# save all the data
df = crime_nearest_school.copy()
if 'time' in df.columns:
    df['time'] = df['time'].astype(str)
    
df.to_parquet('crime_nearest_school.parquet', index=False)
df.to_file('crime_nearest_school.geojson', driver='GeoJSON', index=False)